In [0]:
df = spark.read.parquet("/Volumes/vita/dataengineering/project_data/reviews_parquet/")

In [0]:
df.printSchema()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5517097700776468>, line 1
----> 1 df.printSchema()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2015, in DataFrame.printSchema(self, level)
   2013     print(self.schema.treeString(level))
   2014 else:
-> 2015     print(self.schema.treeString())

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1988, in DataFrame.schema(self)
   1985 @property
   1986 def schema(self) -> StructType:
   1987     # self._schema call will cache the schema and serialize it if it is not cached yet.
-> 1988     _schema = self._schema
   1989     if self._cached_schema_serialized is not None:
   1990         try:

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1977, in DataFrame._schema(self)
   1975 if self._cached_schema is Non

In [0]:
from pyspark.sql.functions import rand
from pyspark.sql.functions import rand, row_number, col
from pyspark.sql.window import Window

# Filter data for years 2014 to 2023
filtered_df = df.filter((col("year") >= 2014) & (col("year") <= 2023))

# Define window partitioned by year and ordered randomly
window_spec = Window.partitionBy("year").orderBy(rand())

# Select 100 random samples from each year
sample_df = (
    filtered_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") <= 100)
    .drop("rn")
)

# Check the number of samples per year
sample_df.groupBy("year").count().orderBy("year").show()

# View the sampled data
display(sample_df)

In [0]:
df.groupBy("rating") \
  .count() \
  .orderBy("rating") \
  .show()

In [0]:
display(df.select("text", "rating").limit(5))

Review Lenghth Analysis

In [0]:
from pyspark.sql.functions import length

df = df.withColumn("review_length", length("text"))

display(df.select("text", "review_length").limit(10))

Count_words

In [0]:
from pyspark.sql.functions import size, split

df = df.withColumn(
    "word_count",
    size(split(df["text"], " "))
)

display(df.select("text", "word_count").limit(10))

Comaparing review lenghth with ratings

In [0]:
from pyspark.sql.functions import avg

display(
    df.groupBy("rating")
      .agg(avg("word_count").alias("Average_Word_Count"))
      .orderBy("rating")
)

Finding shortest review

In [0]:
display(
    df.filter(df.word_count < 3).select("text","word_count")
)


Sentiment Label Generation

The dataset does not contain sentiment labels.

Ratings are converted into sentiment categories:

4-5 Stars → Positive
3 Stars   → Neutral
1-2 Stars → Negative

This creates labels that will be used for sentiment analysis.

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "sentiment",
    when(df.rating <= 2, "Negative")
    .when(df.rating == 3, "Neutral")
    .otherwise("Positive")
)

display(df.select("rating", "sentiment").limit(10))

In [0]:
#Count sentiments
display(
    df.groupBy("sentiment")
      .count()
)

In [0]:
display(
    df.filter(df.sentiment == "Positive")
      .select("text", "rating", "sentiment")
      .limit(10)
)

In [0]:
display(
    df.filter(df.sentiment == "Negative")
      .select("text", "rating", "sentiment")
      .limit(10)
)

In [0]:
#Average review length by sentiment
from pyspark.sql.functions import avg

display(
    df.groupBy("sentiment")
      .agg(avg("word_count").alias("avg_word_count"))
)

In [0]:
from pyspark.sql.functions import col, round

# Total number of reviews
total_reviews = df.count()

# Count and percentage of each sentiment
sentiment_distribution = (
    df.groupBy("sentiment")
      .count()
      .withColumn(
          "percentage",
          round((col("count") / total_reviews) * 100, 2)
      )
      .orderBy("sentiment")
)

# Display the result
display(sentiment_distribution)

Text cleaning
| Step | Purpose                                   |
| ---- | ----------------------------------------- |
| 1    | Convert to lowercase                      |
| 2    | Remove HTML tags                          |
| 3    | Remove URLs                               |
| 4    | Remove special characters and punctuation |
| 5    | Remove numbers (optional)                 |
| 6    | Remove extra spaces                       |
| 7    | Tokenization                              |
| 8    | Remove stop words                         |
| 9    | Lemmatization                             |
| 10   | Generate cleaned text                     |


Convert to lowercase

In [0]:
from pyspark.sql.functions import lower, col

df = df.withColumn("clean_text", lower(col("text")))

display(df.select("text", "clean_text").limit(10))

Step 2: Remove HTML tags

In [0]:

from pyspark.sql.functions import regexp_replace

df = df.withColumn(
    "clean_text",
    regexp_replace("clean_text", "<[^>]+>", "")
)
display(df.select("text", "clean_text").limit(10))

Remove URLs

In [0]:

df = df.withColumn(
    "clean_text",
    regexp_replace("clean_text", r"http\S+|www\S+", "")
)
display(df.select("text", "clean_text").limit(10))


Remove special characters and punctuation

In [0]:

df = df.withColumn(
    "clean_text",
    regexp_replace("clean_text", r"[^a-zA-Z\s]", "")
)
display(df.select("text","clean_text").limit(10))

Remove numbers

In [0]:
df = df.withColumn(
    "clean_text",
    regexp_replace("clean_text", r"\d+", "")
)

display(df.select("text","clean_text").limit(30))                                            

Remove extra space

In [0]:
from pyspark.sql.functions import trim

df = df.withColumn(
    "clean_text",
    trim(regexp_replace("clean_text", "\\s+", " "))
)

Tokenization
Split the review into individual words.

In [0]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="clean_text",
    outputCol="tokens"
)

df = tokenizer.transform(df)

display(df.select("tokens").limit(5))

Remove stop words

In [0]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

df = remover.transform(df)

display(df.select("filtered_tokens").limit(5))

In [0]:
df.printSchema()

Explode the token array

In [0]:
from pyspark.sql.functions import explode

words_df = df.select(explode("filtered_tokens").alias("word"))

display(words_df.limit(10))

Count word frequencies

In [0]:
from pyspark.sql.functions import desc

word_freq = (
    words_df.groupBy("word")
            .count()
            .orderBy(desc("count"))
)

display(word_freq)

View the Top 20 most frequent words

In [0]:
display(
    word_freq.limit(20)
)

Count of Positive Words

In [0]:
from pyspark.sql.functions import explode, col

# List of positive words
positive_words = [
    "good", "great", "excellent", "amazing", "awesome",
    "love", "perfect", "best", "nice", "fantastic",
    "wonderful", "happy", "satisfied", "quality", "recommend"
]

# Explode tokens into individual words
words_df = df.select(explode("filtered_tokens").alias("word"))

# Filter only positive words
positive_df = words_df.filter(col("word").isin(positive_words))

# Count frequency
positive_word_count = (
    positive_df.groupBy("word")
               .count()
               .orderBy(col("count").desc())
)

display(positive_word_count)

In [0]:
negative_words = [
    "bad", "worst", "poor", "terrible", "awful",
    "disappointed", "hate", "broken", "cheap",
    "waste", "defective", "useless", "slow",
    "problem", "issue", "return", "refund"
]

from pyspark.sql.functions import explode, col, desc

# Convert token array into rows
words_df = df.select(explode("filtered_tokens").alias("word"))

# Keep only negative words
negative_df = words_df.filter(col("word").isin(negative_words))

# Count frequency
negative_word_count = (
    negative_df.groupBy("word")
               .count()
               .orderBy(desc("count"))
)

display(negative_word_count)

Calculate Total number of negative words

In [0]:
from pyspark.sql.functions import sum

display(
    negative_word_count.agg(
        sum("count").alias("Total_Negative_Words")
    )
)

In [0]:
from pyspark.sql.functions import sum

display(
    positive_word_count.agg(
        sum("count").alias("Total_Positive_Words")
    )
)

Generate Bigrams

In [0]:
from pyspark.ml.feature import NGram

bigram = NGram(n=2, inputCol="filtered_tokens", outputCol="bigrams")

bigram_df = bigram.transform(df)

display(bigram_df.select("bigrams"))

Bigaram Count

In [0]:
from pyspark.sql.functions import explode, desc

bigram_freq = (
    bigram_df
    .select(explode("bigrams").alias("bigram"))
    .groupBy("bigram")
    .count()
    .orderBy(desc("count"))
)

display(bigram_freq.limit(20))

Trigram Analysis

In [0]:
from pyspark.ml.feature import NGram

trigram = NGram(n=3, inputCol="filtered_tokens", outputCol="trigrams")

trigram_df = trigram.transform(df)

display(trigram_df.select("trigrams"))

Trigram Frequency

In [0]:
from pyspark.sql.functions import explode, desc

trigram_freq = (
    trigram_df
    .select(explode("trigrams").alias("trigram"))
    .groupBy("trigram")
    .count()
    .orderBy(desc("count"))
)

display(trigram_freq.limit(20))

Filter meaningful phrases

In [0]:
display(
    bigram_freq.filter("count >= 10")
)

In [0]:
display(
    trigram_freq.filter("count >= 10")
)

#TF-IDF
Create term frequencies using CountVectorizer


In [0]:
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="rawFeatures",
    vocabSize=20000,
    minDF=5
)

cv_model = cv.fit(df)

cv_df = cv_model.transform(df)

display(cv_df.select("filtered_tokens", "rawFeatures"))

Apply TF-IDF

In [0]:
from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="rawFeatures",
    outputCol="features"
)

idf_model = idf.fit(cv_df)

tfidf_df = idf_model.transform(cv_df)

display(tfidf_df.select("filtered_tokens", "features"))

Check the TF-IDF vectors

In [0]:
display(
   tfidf_df.select("features").limit(10)
)

In [0]:
display(df.limit(10))

In [0]:
display(df.select("product_name").distinct().limit(100))

product wise reviews

In [0]:
from pyspark.sql.functions import count

product_reviews = (
    df.groupBy("product_name")
      .agg(count("*").alias("total_reviews"))
      .orderBy(col("total_reviews").desc())
)

display(product_reviews.limit(10))

Average Rating of Every Product

In [0]:
from pyspark.sql.functions import avg, round

product_rating = (
    df.groupBy("product_name")
      .agg(
          round(avg("rating"),2).alias("average_rating")
      )
      .orderBy(col("average_rating").desc())
)

display(product_rating.limit(20))

Best Selling Products (Approximation)

In [0]:
from pyspark.sql.functions import count

top_products = (
    df.groupBy("product_name")
      .count()
      .orderBy(col("count").desc())
)

display(top_products.limit(20))

Products with Highest Helpful Votes

In [0]:
from pyspark.sql.functions import sum

helpful_products = (
    df.groupBy("product_name")
      .agg(
          sum("helpful_vote").alias("total_helpful_votes")
      )
      .orderBy(col("total_helpful_votes").desc())
)

display(helpful_products.limit(20))

Brand-wise Products

In [0]:
display(df.select("Brand","product_name")\
  .distinct()\
  .orderBy("Brand")\
  .limit(10))
  

Products Under Every Brand

In [0]:
from pyspark.sql.functions import collect_set

brand_products = (
    df.groupBy("Brand")
      .agg(
          collect_set("product_name").alias("Products")
      )
)

display(brand_products)

Top Rated Products (Minimum 100 Reviews)

In [0]:
from pyspark.sql.functions import avg, count, round

best_products = (
    df.groupBy("product_name")
      .agg(
          count("*").alias("reviews"),
          round(avg("rating"),2).alias("avg_rating")
      )
      .filter(col("reviews") >= 100)
      .orderBy(col("avg_rating").desc(), col("reviews").desc())
)

display(best_products.limit(10).orderBy(desc("reviews"),(desc("avg_rating"))))